# Lecture 2: Computational Thinking with Python

## Exercise 2: Chemical Equation Balancer Checker

**Objective:** Write a program that checks if a chemical equation is balanced.

**Problem:** Create a function called `is_balanced()` that takes a chemical equation string and returns True if balanced, False if not.

**Your Task:**
- Split the equation at the arrow symbol "→" to get reactants and products
- For each side, parse the molecules and their coefficients
- Count total atoms of each element on both sides
- Compare the totals

Example Equation Format: "2H2 + O2 → 2H2O"

**Hints:**

- Use equation.split("→") to separate reactants from products
- Use side.split(" + ") to separate individual molecules
- You'll need to handle coefficients (numbers in front of molecules)
- Reuse your count_atoms() function from Exercise 1!
- For molecules with coefficients like "2H2O", the coefficient applies to all atoms in that molecule

**Steps to Think Through:**

- How do you extract the coefficient from "2H2O"? (It's everything before the first letter)
- How do you multiply the atom counts by the coefficient?
- How do you combine counts from multiple molecules on the same side?


In [ ]:
from typing import Dict

def is_balanced(chem_eq: str) -> bool:
    reactants, products = chem_eq.split('→')

    def parse_molecular_formula(formula: str) -> Dict[str, int]:
        atom_dict = {}
        formula = formula.strip()
        i = 0

        # Read the leading coefficient, e.g. 12 in "12H2O".
        while i < len(formula) and formula[i].isdigit():
            i += 1

        coefficient = int(formula[:i]) if i > 0 else 1

        formula = formula[i:].strip()
        i = 0

        while i < len(formula):
            if not formula[i].isupper():
                raise ValueError(f"Expected an element symbol at position {i}")

            atom = formula[i]
            i += 1

            if i < len(formula) and formula[i].islower():
                atom += formula[i]
                i += 1

            start = i
            while i < len(formula) and formula[i].isdigit():
                i += 1

            count = int(formula[start:i]) if i > start else 1

            atom_dict[atom] = (
                atom_dict.get(atom, 0) + coefficient * count
            )

        return atom_dict

    reactant_counts = {}
    for reactant in reactants.split('+'):
        for element, count in parse_molecular_formula(reactant).items():
            reactant_counts[element] = reactant_counts.get(element, 0) + count

    product_counts = {}
    for product in products.split('+'):
        for element, count in parse_molecular_formula(product).items():
            product_counts[element] = product_counts.get(element, 0) + count

    return reactant_counts == product_counts



### Test Cases


In [2]:
# If the outcome is incorrect, the keyword assert will cause an error

print(is_balanced("2H2 + O2 → 2H2O"))
assert is_balanced("2H2 + O2 → 2H2O")

print(is_balanced("H2 + O2 → H2O"))
assert not is_balanced("H2 + O2 → H2O")

print(is_balanced("CH4 + 2O2 → CO2 + 2H2O"))
assert is_balanced("CH4 + 2O2 → CO2 + 2H2O")

print(is_balanced("N2 + H2 → NH3"))
assert not is_balanced("N2 + H2 → NH3")

True
False
True
False


## Exercise 3: Reaction Pathway Analyzer

**Objective:** Analyze multi-step reaction sequences to find synthesis pathways.

**Problem:** Write a program that determines what compounds can be synthesized from starting materials through a series of reactions, and finds a pathway to create a target compound.

**Your Task:**
Create a function called `find_synthesis_path()` that takes:

- A list of starting compounds
- A list of reaction equations
- A target compound to synthesize

The function should return either:

- A list of reaction steps that lead to the target compound, OR
- A message saying the target cannot be synthesized

**Hints:**

- Start by creating a set of "available" compounds (your starting materials)
- For each reaction, check if you have all the reactants needed
- If you can perform a reaction, add the products to your available compounds
- Keep track of which reactions you've used in what order
- Continue until you either find the target or can't make any new compounds

#### Computational Thinking:

**Decomposition:** Break this into smaller problems

- Parse reaction equations
- Track available compounds
- Check if reactions can be performed
- Record the synthesis pathway


**Pattern Recognition:** What patterns do you see?

- Each reaction has the same format: "reactants → products"
- You need to repeatedly check if new reactions become possible


**Algorithm Design:** What's your step-by-step approach?

- Initialize available compounds
- Loop: try each reaction, see if possible, update available compounds
- Stop when target is found or no progress is made


**Abstraction:** What functions might be helpful?

- A function to parse a single reaction equation
- A function to check if a reaction is possible with current compounds
- A function to update available compounds after a reaction



In [3]:
from typing import List

def find_synthesis_path(
    starting_materials: List[str],
    reactions: List[str],
    target: str
) -> List[str] | str:

    def parse_side(side: str) -> set[str]:
        compounds = set()

        for term in side.split("+"):
            term = term.strip()

            i = 0
            while i < len(term) and term[i].isdigit():
                i += 1

            compounds.add(term[i:].strip())

        return compounds

    available = {compound.strip() for compound in starting_materials}
    target = target.strip()
    pathway = []

    if target in available:
        return pathway

    parsed_reactions = []
    for reaction in reactions:
        left, right = reaction.split("→")
        parsed_reactions.append(
            (reaction, parse_side(left), parse_side(right))
        )

    while True:
        progress = False

        for reaction, reactants, products in parsed_reactions:
        
            if reactants.issubset(available):

            
                if products - available:
                    available.update(products)
                    pathway.append(reaction)
                    progress = True

                    if target in available:
                        return pathway

    
        if not progress:
            return f"{target} cannot be synthesized."

### Test Cases

In [4]:
# Basic Multi-Step Synthesis

starting_materials = ["H2", "O2", "N2"]
reactions = [
    "2H2 + O2 → 2H2O",
    "N2 + 3H2 → 2NH3", 
    "NH3 + H2O → NH4OH"
]
target = "NH4OH"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find a path using H2 + N2 → NH3, then NH3 + H2O → NH4OH

['2H2 + O2 → 2H2O', 'N2 + 3H2 → 2NH3', 'NH3 + H2O → NH4OH']


In [5]:
# Direct Synthesis (One Step)

starting_materials = ["Na", "Cl2"]
reactions = [
    "2Na + Cl2 → 2NaCl",
    "NaCl + H2O → NaOH + HCl"
]
target = "NaCl"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find direct synthesis in one step

['2Na + Cl2 → 2NaCl']


In [6]:
# Impossible Synthesis

starting_materials = ["H2", "O2"]
reactions = [
    "2H2 + O2 → 2H2O",
    "N2 + 3H2 → 2NH3"  # But we don't have N2!
]
target = "NH3"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should return message that NH3 cannot be synthesized

NH3 cannot be synthesized.


In [7]:
# Multiple Pathway Options
starting_materials = ["C", "H2", "O2", "H2O"]
reactions = [
    "C + O2 → CO2",
    "2H2 + O2 → 2H2O",
    "CO2 + H2O → H2CO3",
    "C + 2H2 → CH4",
    "CH4 + 2O2 → CO2 + 2H2O"
]
target = "CO2"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find one of multiple possible paths to CO2

['C + O2 → CO2']


In [8]:
# Long Chain Synthesis
starting_materials = ["Fe", "O2", "C"]
reactions = [
    "4Fe + 3O2 → 2Fe2O3",
    "2C + O2 → 2CO",
    "Fe2O3 + 3CO → 2Fe + 3CO2",
    "CO2 + C → 2CO"
]
target = "CO2"

result = find_synthesis_path(starting_materials, reactions, target)
print(result)
# Expected: Should find the multi-step path involving iron oxide formation and reduction

['4Fe + 3O2 → 2Fe2O3', '2C + O2 → 2CO', 'Fe2O3 + 3CO → 2Fe + 3CO2']
